# Análisis Exploratorio de Datos (EDA) - Dataset Ecommify/Olist

## Objetivo
Conducir un **EDA exhaustivo** sobre el dataset de E-commerce público de Olist (Ecommify) para:
- Entender la estructura, volumen y relaciones de los datos
- Evaluar la calidad e identificar inconsistencias
- Analizar patrones univariados y bivariados
- Diseñar arquitectura híbrida PostgreSQL + MongoDB
- Generar matriz de decisiones arquitectónicas

## Archivos del Dataset
1. `olist_customers_dataset.csv` - Información de clientes y ubicación
2. `olist_geolocation_dataset.csv` - Coordenadas geográficas por código postal
3. `olist_order_items_dataset.csv` - Detalle de artículos en cada orden
4. `olist_order_payments_dataset.csv` - Métodos de pago y montos
5. `olist_order_reviews_dataset.csv` - Reseñas, calificaciones y comentarios
6. `olist_orders_dataset.csv` - Ciclo de vida de las órdenes y timestamps
7. `olist_products_dataset.csv` - Atributos físicos y categorías de productos
8. `olist_sellers_dataset.csv` - Información de vendedores y ubicación
9. `product_category_name_translation.csv` - Traducción de nombres de categorías

In [13]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [14]:
# Load All Datasets
data_path = Path.cwd()

# Load CSV files
df_customers = pd.read_csv(data_path / 'olist_customers_dataset.csv')
df_geolocation = pd.read_csv(data_path / 'olist_geolocation_dataset.csv')
df_order_items = pd.read_csv(data_path / 'olist_order_items_dataset.csv')
df_order_payments = pd.read_csv(data_path / 'olist_order_payments_dataset.csv')
df_order_reviews = pd.read_csv(data_path / 'olist_order_reviews_dataset.csv')
df_orders = pd.read_csv(data_path / 'olist_orders_dataset.csv')
df_products = pd.read_csv(data_path / 'olist_products_dataset.csv')
df_sellers = pd.read_csv(data_path / 'olist_sellers_dataset.csv')
df_category_translation = pd.read_csv(data_path / 'product_category_name_translation.csv')

# Store all dataframes in a dictionary for easy access
datasets = {
    'customers': df_customers,
    'geolocation': df_geolocation,
    'order_items': df_order_items,
    'order_payments': df_order_payments,
    'order_reviews': df_order_reviews,
    'orders': df_orders,
    'products': df_products,
    'sellers': df_sellers,
    'category_translation': df_category_translation
}

print("✓ All datasets loaded successfully")
print(f"\nDatasets available: {list(datasets.keys())}")

✓ All datasets loaded successfully

Datasets available: ['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'sellers', 'category_translation']


## Paso 1: Entendimiento Estructural, Volumen y Mapeo de Entidades

### Resumen de Datasets
Resumen inicial del número de registros y estructura básica de cada tabla.

In [15]:
# Resumen de Volumen de Datasets - PASO 1
summary_data = []
for name, df in datasets.items():
    summary_data.append({
        'Tabla': name,
        'Registros': df.shape[0],
        'Columnas': df.shape[1],
        'Memoria (MB)': df.memory_usage(deep=True).sum() / 1024**2
    })

summary_df = pd.DataFrame(summary_data)
print("=" * 80)
print("PASO 1: ENTENDIMIENTO ESTRUCTURAL Y RESUMEN DE VOLUMEN")
print("=" * 80)
print(summary_df.to_string(index=False))
print(f"\nTotal de Registros en Todas las Tablas: {summary_df['Registros'].sum():,}")
print(f"Uso Total de Memoria: {summary_df['Memoria (MB)'].sum():.2f} MB")

PASO 1: ENTENDIMIENTO ESTRUCTURAL Y RESUMEN DE VOLUMEN
               Tabla  Registros  Columnas  Memoria (MB)
           customers      99441         5     29.621103
         geolocation    1000163         5    145.202189
         order_items     112650         7     39.427454
      order_payments     103886         5     17.814588
       order_reviews      99224         7     42.746946
              orders      99441         8     58.969229
            products      32951         9      6.794703
             sellers       3095         4      0.658910
category_translation         71         2      0.010082

Total de Registros en Todas las Tablas: 1,550,922
Uso Total de Memoria: 341.25 MB


In [16]:
# Análisis Detallado de Estructura - PASO 1
print("\n" + "=" * 80)
print("ESTRUCTURAS DETALLADAS DE TABLAS (df.info())")
print("=" * 80)

for name, df in datasets.items():
    print(f"\n--- {name.upper()} ---")
    print(f"Forma: {df.shape[0]} filas × {df.shape[1]} columnas")
    df.info()


ESTRUCTURAS DETALLADAS DE TABLAS (df.info())

--- CUSTOMERS ---
Forma: 99441 filas × 5 columnas
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB

--- GEOLOCATION ---
Forma: 1000163 filas × 5 columnas
<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  f

## Paso 2: Análisis de Calidad y Sanidad de los Datos

### Evaluación de Nulos, Duplicados y Cardinalidad
Análisis detallado de la integridad de datos y valores faltantes en cada tabla.

In [17]:
# Análisis de Calidad de Datos - PASO 2
print("\n" + "=" * 80)
print("PASO 2: ANÁLISIS DE CALIDAD Y SANIDAD DE LOS DATOS")
print("=" * 80)

quality_report = []
for name, df in datasets.items():
    null_count = df.isnull().sum().sum()
    null_pct = (null_count / (df.shape[0] * df.shape[1])) * 100
    duplicates = df.duplicated().sum()
    
    quality_report.append({
        'Tabla': name,
        'Total Nulos': null_count,
        '% Nulos': f"{null_pct:.2f}%",
        'Duplicados': duplicates,
        'Valores Únicos (PK Candidate)': f"{df.iloc[:, 0].nunique()} / {df.shape[0]}"
    })

quality_df = pd.DataFrame(quality_report)
print("\n📊 RESUMEN DE CALIDAD DE DATOS:")
print(quality_df.to_string(index=False))

# Análisis detallado de nulos por columna
print("\n\n📍 PORCENTAJE DE NULOS POR COLUMNA:")
print("-" * 80)
for name, df in datasets.items():
    null_pct_by_col = (df.isnull().sum() / len(df) * 100).round(2)
    if null_pct_by_col.sum() > 0:
        print(f"\n{name.upper()}:")
        for col, pct in null_pct_by_col[null_pct_by_col > 0].items():
            print(f"  • {col}: {pct}%")
    else:
        print(f"\n{name.upper()}: ✓ Sin valores nulos")


PASO 2: ANÁLISIS DE CALIDAD Y SANIDAD DE LOS DATOS

📊 RESUMEN DE CALIDAD DE DATOS:
               Tabla  Total Nulos % Nulos  Duplicados Valores Únicos (PK Candidate)
           customers            0   0.00%           0                 99441 / 99441
         geolocation            0   0.00%      261831               19015 / 1000163
         order_items            0   0.00%           0                98666 / 112650
      order_payments            0   0.00%           0                99440 / 103886
       order_reviews       145903  21.01%           0                 98410 / 99224
              orders         4908   0.62%           0                 99441 / 99441
            products         2448   0.83%           0                 32951 / 32951
             sellers            0   0.00%           0                   3095 / 3095
category_translation            0   0.00%           0                       71 / 71


📍 PORCENTAJE DE NULOS POR COLUMNA:
---------------------------------------

## Paso 3: Análisis Univariado (Métricas Clave)

### Estadísticas Descriptivas y Distribuciones
Análisis detallado de variables numéricas y categóricas por tabla.

In [8]:
# Análisis Univariado - PASO 3
print("\n" + "=" * 80)
print("PASO 3: ANÁLISIS UNIVARIADO - MÉTRICAS CLAVE")
print("=" * 80)

# Análisis de variables numéricas clave
print("\n📈 VARIABLES NUMÉRICAS - ESTADÍSTICAS DESCRIPTIVAS:")
print("-" * 80)

numeric_vars = {
    'order_items': ['price', 'freight_value'],
    'order_payments': ['payment_value'],
    'order_reviews': ['review_score'],
    'products': ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
}

for table, columns in numeric_vars.items():
    if table in datasets:
        df = datasets[table]
        print(f"\n{table.upper()}:")
        for col in columns:
            if col in df.columns:
                # Obtener estadísticas numéricas
                mean_val = df[col].mean()
                median_val = df[col].median()
                std_val = df[col].std()
                min_val = df[col].min()
                max_val = df[col].max()
                
                print(f"\n  {col}:")
                print(f"    Media: {mean_val:.2f}")
                print(f"    Mediana: {median_val:.2f}")
                print(f"    Desv. Est.: {std_val:.2f}")
                print(f"    Mín: {min_val:.2f}, Máx: {max_val:.2f}")
                
                # Detectar outliers (IQR method)
                Q1 = df[col].quantile(0.25)
                Q3 = df[col].quantile(0.75)
                IQR = Q3 - Q1
                outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
                print(f"    Outliers detectados: {outliers} ({(outliers/len(df)*100):.2f}%)")

# Análisis de variables categóricas clave
print("\n\n🏷️  VARIABLES CATEGÓRICAS - FRECUENCIAS:")
print("-" * 80)

categorical_vars = {
    'orders': ['order_status'],
    'order_payments': ['payment_type'],
    'order_reviews': ['review_score'],
    'products': ['product_category_name'],
    'customers': ['customer_state']
}

for table, columns in categorical_vars.items():
    if table in datasets:
        df = datasets[table]
        print(f"\n{table.upper()}:")
        for col in columns:
            if col in df.columns:
                print(f"\n  {col} (Top 10):")
                top_vals = df[col].value_counts().head(10)
                for idx, (val, count) in enumerate(top_vals.items(), 1):
                    pct = (count / len(df)) * 100
                    print(f"    {idx}. {val}: {count:,} ({pct:.1f}%)")


PASO 3: ANÁLISIS UNIVARIADO - MÉTRICAS CLAVE

📈 VARIABLES NUMÉRICAS - ESTADÍSTICAS DESCRIPTIVAS:
--------------------------------------------------------------------------------

ORDER_ITEMS:

  price:
    Media: 120.65
    Mediana: 74.99
    Desv. Est.: 183.63
    Mín: 0.85, Máx: 6735.00
    Outliers detectados: 8427 (7.48%)

  freight_value:
    Media: 19.99
    Mediana: 16.26
    Desv. Est.: 15.81
    Mín: 0.00, Máx: 409.68
    Outliers detectados: 12134 (10.77%)

ORDER_PAYMENTS:

  payment_value:
    Media: 154.10
    Mediana: 100.00
    Desv. Est.: 217.49
    Mín: 0.00, Máx: 13664.08
    Outliers detectados: 7981 (7.68%)

ORDER_REVIEWS:

  review_score:
    Media: 4.09
    Mediana: 5.00
    Desv. Est.: 1.35
    Mín: 1.00, Máx: 5.00
    Outliers detectados: 14575 (14.69%)

PRODUCTS:

  product_weight_g:
    Media: 2276.47
    Mediana: 700.00
    Desv. Est.: 4282.04
    Mín: 0.00, Máx: 40425.00
    Outliers detectados: 4551 (13.81%)

  product_length_cm:
    Media: 30.82
    Median

## Paso 4: Análisis Bivariado, Relacional y Patrones de Acceso

### Relaciones de Negocio e Identificación de Llaves
Análisis de cómo interactúan las tablas e identificación de llaves primarias/foráneas para arquitectura híbrida.

In [9]:
# Análisis Bivariado y Mapeo Arquitectónico - PASO 4
print("\n" + "=" * 80)
print("PASO 4: ANÁLISIS BIVARIADO Y MAPEO PARA ARQUITECTURA HÍBRIDA")
print("=" * 80)

# Identificar relaciones clave entre tablas
print("\n🔗 RELACIONES IDENTIFICADAS (Llaves Primarias y Foráneas):")
print("-" * 80)

relationships = {
    'customers': {
        'pk': 'customer_id',
        'description': 'Centro de datos de clientes',
        'fk_to': ['geolocation (customer_zip_code_prefix)']
    },
    'orders': {
        'pk': 'order_id',
        'description': 'Transacciones principales',
        'fk_to': ['customers (customer_id)', 'order_items (order_id)', 'order_payments (order_id)', 'order_reviews (order_id)']
    },
    'order_items': {
        'pk': 'order_item_id',
        'description': 'Detalles de artículos en órdenes',
        'fk_to': ['orders (order_id)', 'products (product_id)', 'sellers (seller_id)']
    },
    'order_payments': {
        'pk': 'N/A (payment_sequential + order_id)',
        'description': 'Métodos y montos de pago',
        'fk_to': ['orders (order_id)']
    },
    'order_reviews': {
        'pk': 'review_id',
        'description': 'Reseñas y calificaciones',
        'fk_to': ['orders (order_id)']
    },
    'products': {
        'pk': 'product_id',
        'description': 'Catálogo de productos',
        'fk_to': ['category_translation (product_category_name)']
    },
    'sellers': {
        'pk': 'seller_id',
        'description': 'Información de vendedores',
        'fk_to': ['geolocation (seller_zip_code_prefix)']
    },
    'geolocation': {
        'pk': 'geolocation_zip_code_prefix (non-unique)',
        'description': 'Datos geográficos',
        'fk_to': []
    }
}

for table, info in relationships.items():
    print(f"\n{table.upper()}:")
    print(f"  PK: {info['pk']}")
    print(f"  Descripción: {info['description']}")
    print(f"  FK: {', '.join(info['fk_to']) if info['fk_to'] else 'Ninguna'}")

# Análisis de patrones de acceso y volumen
print("\n\n📊 PATRONES DE ACCESO Y VOLUMEN:")
print("-" * 80)

# Verificar relaciones en datos
print("\nVerificación de Integridad Referencial:")

# Clientes únicos en órdenes
unique_customers_in_orders = datasets['orders']['customer_id'].nunique()
total_customers = datasets['customers'].shape[0]
print(f"\n  • Clientes en órdenes: {unique_customers_in_orders:,} de {total_customers:,} ({unique_customers_in_orders/total_customers*100:.1f}%)")

# Órdenes con múltiples artículos
orders_with_items = datasets['order_items']['order_id'].nunique()
print(f"  • Órdenes con artículos: {orders_with_items:,} de {datasets['orders'].shape[0]:,}")

# Vendedores únicos
unique_sellers = datasets['order_items']['seller_id'].nunique()
total_sellers = datasets['sellers'].shape[0]
print(f"  • Vendedores activos: {unique_sellers:,} de {total_sellers:,} ({unique_sellers/total_sellers*100:.1f}%)")

# Productos en catálogo vs vendidos
products_in_orders = datasets['order_items']['product_id'].nunique()
total_products = datasets['products'].shape[0]
print(f"  • Productos vendidos: {products_in_orders:,} de {total_products:,} ({products_in_orders/total_products*100:.1f}%)")

# Reseñas por orden
orders_with_reviews = datasets['order_reviews']['order_id'].nunique()
print(f"  • Órdenes con reseñas: {orders_with_reviews:,} de {datasets['orders'].shape[0]:,} ({orders_with_reviews/datasets['orders'].shape[0]*100:.1f}%)")


PASO 4: ANÁLISIS BIVARIADO Y MAPEO PARA ARQUITECTURA HÍBRIDA

🔗 RELACIONES IDENTIFICADAS (Llaves Primarias y Foráneas):
--------------------------------------------------------------------------------

CUSTOMERS:
  PK: customer_id
  Descripción: Centro de datos de clientes
  FK: geolocation (customer_zip_code_prefix)

ORDERS:
  PK: order_id
  Descripción: Transacciones principales
  FK: customers (customer_id), order_items (order_id), order_payments (order_id), order_reviews (order_id)

ORDER_ITEMS:
  PK: order_item_id
  Descripción: Detalles de artículos en órdenes
  FK: orders (order_id), products (product_id), sellers (seller_id)

ORDER_PAYMENTS:
  PK: N/A (payment_sequential + order_id)
  Descripción: Métodos y montos de pago
  FK: orders (order_id)

ORDER_REVIEWS:
  PK: review_id
  Descripción: Reseñas y calificaciones
  FK: orders (order_id)

PRODUCTS:
  PK: product_id
  Descripción: Catálogo de productos
  FK: category_translation (product_category_name)

SELLERS:
  PK: seller_

## Paso 5: Análisis Temporal, Geográfico y Ciclo de Vida

### Distribuciones Temporales y Geográficas
Análisis de tendencias de compra, estacionalidad y concentración geográfica.

In [10]:
# Análisis Temporal, Geográfico y Ciclo de Vida - PASO 5
print("\n" + "=" * 80)
print("PASO 5: ANÁLISIS TEMPORAL, GEOGRÁFICO Y CICLO DE VIDA")
print("=" * 80)

# Análisis temporal
print("\n⏱️  ANÁLISIS TEMPORAL:")
print("-" * 80)

# Convertir timestamps
df_orders = datasets['orders'].copy()
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
df_orders['order_delivered_customer_date'] = pd.to_datetime(df_orders['order_delivered_customer_date'])
df_orders['order_estimated_delivery_date'] = pd.to_datetime(df_orders['order_estimated_delivery_date'])

print(f"\nRango de fechas de compra:")
print(f"  Inicio: {df_orders['order_purchase_timestamp'].min()}")
print(f"  Fin: {df_orders['order_purchase_timestamp'].max()}")
print(f"  Duración: {(df_orders['order_purchase_timestamp'].max() - df_orders['order_purchase_timestamp'].min()).days} días")

# Calcular tiempo de entrega
df_orders['delivery_time_days'] = (df_orders['order_delivered_customer_date'] - 
                                    df_orders['order_purchase_timestamp']).dt.days
df_orders['estimated_time_days'] = (df_orders['order_estimated_delivery_date'] - 
                                     df_orders['order_purchase_timestamp']).dt.days
df_orders['delivery_delay_days'] = df_orders['delivery_time_days'] - df_orders['estimated_time_days']

print(f"\nTiempo de Entrega (días):")
print(f"  Media: {df_orders['delivery_time_days'].mean():.1f} días")
print(f"  Mediana: {df_orders['delivery_time_days'].median():.1f} días")
print(f"  Mín-Máx: {df_orders['delivery_time_days'].min():.0f} - {df_orders['delivery_time_days'].max():.0f} días")

print(f"\nRetraso de Entrega (vs estimado):")
print(f"  Media: {df_orders['delivery_delay_days'].mean():.1f} días")
print(f"  Órdenes retrasadas: {(df_orders['delivery_delay_days'] > 0).sum():,} ({(df_orders['delivery_delay_days'] > 0).sum()/len(df_orders)*100:.1f}%)")

# Estados de órdenes
print(f"\nEstados de Órdenes:")
order_status = df_orders['order_status'].value_counts()
for status, count in order_status.items():
    pct = (count / len(df_orders)) * 100
    print(f"  • {status}: {count:,} ({pct:.1f}%)")

# Análisis geográfico
print("\n\n🌍 ANÁLISIS GEOGRÁFICO:")
print("-" * 80)

# Distribución de clientes por estado
print(f"\nDistribución de Clientes por Estado (Top 15):")
customer_state = datasets['customers']['customer_state'].value_counts().head(15)
for idx, (state, count) in enumerate(customer_state.items(), 1):
    pct = (count / datasets['customers'].shape[0]) * 100
    print(f"  {idx:2d}. {state}: {count:,} ({pct:.1f}%)")

# Distribución de órdenes por estado
print(f"\nDistribución de Órdenes por Estado (Top 15):")
orders_by_state = df_orders.merge(datasets['customers'][['customer_id', 'customer_state']], 
                                   on='customer_id')['customer_state'].value_counts().head(15)
for idx, (state, count) in enumerate(orders_by_state.items(), 1):
    pct = (count / df_orders.shape[0]) * 100
    print(f"  {idx:2d}. {state}: {count:,} ({pct:.1f}%)")

# Concentración geográfica
print(f"\nConcentración Geográfica:")
top_3_states = orders_by_state.head(3).sum()
print(f"  Top 3 estados: {top_3_states:,} órdenes ({top_3_states/df_orders.shape[0]*100:.1f}% del total)")

# Ciudades con coordenadas
print(f"\nCobertura Geográfica:")
print(f"  Códigos postales únicos en geolocation: {datasets['geolocation']['geolocation_zip_code_prefix'].nunique():,}")
print(f"  Clientes únicos: {datasets['customers'].shape[0]:,}")

# Análisis de ciclo de vida de productos
print("\n\n🔄 CICLO DE VIDA Y ACTIVIDAD:")
print("-" * 80)

print(f"\nOrdenes por categoría de producto (Top 15):")
orders_by_category = datasets['order_items'].merge(
    datasets['products'][['product_id', 'product_category_name']], 
    on='product_id'
)['product_category_name'].value_counts().head(15)

for idx, (category, count) in enumerate(orders_by_category.items(), 1):
    avg_price = datasets['order_items'].merge(
        datasets['products'][['product_id', 'product_category_name']], 
        on='product_id'
    )[datasets['order_items'].merge(
        datasets['products'][['product_id', 'product_category_name']], 
        on='product_id'
    )['product_category_name'] == category]['price'].mean()
    print(f"  {idx:2d}. {category}: {count:,} items (Precio promedio: R${avg_price:.2f})")

print(f"\nMétodos de pago más utilizados:")
payment_type = datasets['order_payments']['payment_type'].value_counts()
for payment, count in payment_type.items():
    pct = (count / datasets['order_payments'].shape[0]) * 100
    print(f"  • {payment}: {count:,} ({pct:.1f}%)")


PASO 5: ANÁLISIS TEMPORAL, GEOGRÁFICO Y CICLO DE VIDA

⏱️  ANÁLISIS TEMPORAL:
--------------------------------------------------------------------------------

Rango de fechas de compra:
  Inicio: 2016-09-04 21:15:19
  Fin: 2018-10-17 17:30:18
  Duración: 772 días

Tiempo de Entrega (días):
  Media: 12.1 días
  Mediana: 10.0 días
  Mín-Máx: 0 - 209 días

Retraso de Entrega (vs estimado):
  Media: -11.3 días
  Órdenes retrasadas: 7,308 (7.3%)

Estados de Órdenes:
  • delivered: 96,478 (97.0%)
  • shipped: 1,107 (1.1%)
  • canceled: 625 (0.6%)
  • unavailable: 609 (0.6%)
  • invoiced: 314 (0.3%)
  • processing: 301 (0.3%)
  • created: 5 (0.0%)
  • approved: 2 (0.0%)


🌍 ANÁLISIS GEOGRÁFICO:
--------------------------------------------------------------------------------

Distribución de Clientes por Estado (Top 15):
   1. SP: 41,746 (42.0%)
   2. RJ: 12,852 (12.9%)
   3. MG: 11,635 (11.7%)
   4. RS: 5,466 (5.5%)
   5. PR: 5,045 (5.1%)
   6. SC: 3,637 (3.7%)
   7. BA: 3,380 (3.4%)
   8. 

## Matriz de Decisiones Arquitectónicas: PostgreSQL vs MongoDB

### Recomendaciones para Arquitectura Híbrida SQL + NoSQL

In [11]:
# Matriz de Decisiones Arquitectónicas - PostgreSQL vs MongoDB
print("\n" + "=" * 100)
print("MATRIZ DE DECISIONES ARQUITECTÓNICAS: PostgreSQL vs MongoDB")
print("=" * 100)

architecture_matrix = {
    'POSTGRESQL (SQL Relacional)': {
        'Entidades': [
            'customers - Clientes (PK: customer_id)',
            'orders - Transacciones (PK: order_id) **CRÍTICO**',
            'order_payments - Pagos (PK: order_id + payment_sequential) **CRÍTICO**',
            'sellers - Vendedores (PK: seller_id)',
            'geolocation - Ubicaciones geográficas'
        ],
        'Justificación': [
            '✓ ACID transaccional requerido para órdenes y pagos',
            '✓ Integridad referencial crítica (clientes → órdenes → pagos)',
            '✓ Normalización de datos para evitar redundancia',
            '✓ Consultas complejas con JOINs (reportes financieros)',
            '✓ Auditoría y trazabilidad de transacciones',
            '✓ Alta consistencia requerida'
        ],
        'Patrones de Acceso': [
            '→ Lectura-Escritura frecuente (transacciones en tiempo real)',
            '→ Consultas complejas con agregaciones',
            '→ Acceso por FK (customer_id, seller_id, order_id)',
            '→ Reportes analíticos con JOINs múltiples'
        ],
        'Restricciones': 'Límite de 16MB no aplica. Indexación por customer_id, order_id, order_status'
    },
    'MONGODB (NoSQL Documento)': {
        'Entidades': [
            'products - Catálogo (denormalizado con categorías)',
            'order_items - Detalles de compra (embebidos en Order Document)',
            'order_reviews - Reseñas (embebidas en Order Document)',
            'product_category_translation - Categorías (denormalizadas)'
        ],
        'Justificación': [
            '✓ Lecturas rápidas de catálogo (producto + categoría + metadata)',
            '✓ Estructura de documento flexible (atributos de producto variables)',
            '✓ Desnormalización beneficiosa para queries de lectura intensiva',
            '✓ Embebimiento de orden con items y reseñas (queries atómicas)',
            '✓ Alto rendimiento para reportes de reseñas y rating por producto',
            '✓ Escalabilidad horizontal con sharding por product_id o order_id'
        ],
        'Patrones de Acceso': [
            '→ Lectura intensiva (página de producto, catálogo)',
            '→ Agregaciones de datos relacionados (orden + items + reseña)',
            '→ Queries por categoría o atributos de producto',
            '→ Búsqueda por palabras clave en descripción o categoría'
        ],
        'Restricciones': 'Documento < 16MB. Si una orden con todos sus items + reseña > 16MB, mantener en PostgreSQL'
    }
}

for db_type, details in architecture_matrix.items():
    print(f"\n{'🔵' if 'POSTGRESQL' in db_type else '🟢'} {db_type}")
    print("-" * 100)
    
    print(f"\n  📊 Entidades Recomendadas:")
    for entity in details['Entidades']:
        print(f"     • {entity}")
    
    print(f"\n  📈 Justificación:")
    for reason in details['Justificación']:
        print(f"     {reason}")
    
    print(f"\n  🔍 Patrones de Acceso Esperados:")
    for pattern in details['Patrones de Acceso']:
        print(f"     {pattern}")
    
    print(f"\n  ⚠️  Restricciones y Consideraciones:")
    print(f"     {details['Restricciones']}")

# Resumen de índices recomendados
print("\n\n" + "=" * 100)
print("RECOMENDACIONES DE ÍNDICES")
print("=" * 100)

indexes = {
    'PostgreSQL': {
        'customers': ['customer_id (PK)', 'customer_state', 'customer_city'],
        'orders': ['order_id (PK)', 'customer_id (FK)', 'order_status', 'order_purchase_timestamp'],
        'order_payments': ['order_id (FK)', 'payment_type'],
        'sellers': ['seller_id (PK)', 'seller_state'],
        'order_items': ['order_id (FK)', 'seller_id (FK)', 'product_id (FK)']
    },
    'MongoDB': {
        'products': ['product_id (PK)', 'product_category_name', 'product_weight_g'],
        'orders_denormalized': ['order_id (PK)', 'customer_id', 'order_purchase_timestamp', 'items.product_id']
    }
}

for db, tables in indexes.items():
    print(f"\n{db}:")
    for table, index_list in tables.items():
        print(f"  {table}:")
        for idx in index_list:
            print(f"    • {idx}")

print("\n" + "=" * 100)
print("✅ ANÁLISIS EDA COMPLETADO")
print("=" * 100)
print("\nPróximos pasos:")
print("1. Crear esquema PostgreSQL con las tablas normalizadas")
print("2. Crear colecciones MongoDB con estructura de documentos")
print("3. Implementar ETL para sincronizar datos entre sistemas")
print("4. Definir estrategia de replicación y consistencia eventual")
print("5. Configurar índices y optimizar queries según patrones de acceso")


MATRIZ DE DECISIONES ARQUITECTÓNICAS: PostgreSQL vs MongoDB

🔵 POSTGRESQL (SQL Relacional)
----------------------------------------------------------------------------------------------------

  📊 Entidades Recomendadas:
     • customers - Clientes (PK: customer_id)
     • orders - Transacciones (PK: order_id) **CRÍTICO**
     • order_payments - Pagos (PK: order_id + payment_sequential) **CRÍTICO**
     • sellers - Vendedores (PK: seller_id)
     • geolocation - Ubicaciones geográficas

  📈 Justificación:
     ✓ ACID transaccional requerido para órdenes y pagos
     ✓ Integridad referencial crítica (clientes → órdenes → pagos)
     ✓ Normalización de datos para evitar redundancia
     ✓ Consultas complejas con JOINs (reportes financieros)
     ✓ Auditoría y trazabilidad de transacciones
     ✓ Alta consistencia requerida

  🔍 Patrones de Acceso Esperados:
     → Lectura-Escritura frecuente (transacciones en tiempo real)
     → Consultas complejas con agregaciones
     → Acceso por FK 

## Diccionario de Datos Unificado

Resumen de columnas, tipos de datos, valores nulos y roles en el modelo conceptual.

In [12]:
# Diccionario de Datos Unificado
print("\n" + "=" * 120)
print("DICCIONARIO DE DATOS UNIFICADO")
print("=" * 120)

data_dict = []

for table_name, df in datasets.items():
    for col in df.columns:
        null_count = df[col].isnull().sum()
        null_pct = (null_count / len(df)) * 100
        dtype = df[col].dtype
        
        # Determinar rol
        if col.endswith('_id') and not col.endswith('_id_name'):
            if col == df.columns[0]:
                role = 'PK (Llave Primaria)'
            else:
                role = 'FK (Llave Foránea)'
        elif 'timestamp' in col.lower() or 'date' in col.lower():
            role = 'Atributo Temporal'
        elif 'value' in col.lower() or 'price' in col.lower() or col.endswith('_score'):
            role = 'Atributo Medible'
        else:
            role = 'Atributo Descriptivo'
        
        # Obtener cardinalidad
        unique_count = df[col].nunique()
        
        data_dict.append({
            'Tabla': table_name,
            'Columna': col,
            'Tipo': str(dtype),
            'Nulos': f"{null_count} ({null_pct:.1f}%)",
            'Cardinalidad': unique_count,
            'Rol': role
        })

data_dict_df = pd.DataFrame(data_dict)

# Mostrar por tabla
for table_name in datasets.keys():
    table_data = data_dict_df[data_dict_df['Tabla'] == table_name]
    print(f"\n📋 {table_name.upper()}")
    print("-" * 120)
    print(table_data[['Columna', 'Tipo', 'Nulos', 'Cardinalidad', 'Rol']].to_string(index=False))

print("\n\n" + "=" * 120)
print("LEYENDA DE ROLES")
print("=" * 120)
print("""
  🔵 PK (Llave Primaria)      → Identificador único de registro
  🟢 FK (Llave Foránea)       → Referencia a otra tabla
  ⚫ Atributo Temporal         → Timestamp, fecha, duración
  🟡 Atributo Medible         → Valores numéricos (precios, peso, puntuación)
  ⚪ Atributo Descriptivo     → Texto, categoría, estado
""")

print("\n✅ EDA COMPLETADO - Todos los datos están listos para arquitectura PostgreSQL + MongoDB")


DICCIONARIO DE DATOS UNIFICADO

📋 CUSTOMERS
------------------------------------------------------------------------------------------------------------------------
                 Columna  Tipo    Nulos  Cardinalidad                  Rol
             customer_id   str 0 (0.0%)         99441  PK (Llave Primaria)
      customer_unique_id   str 0 (0.0%)         96096   FK (Llave Foránea)
customer_zip_code_prefix int64 0 (0.0%)         14994 Atributo Descriptivo
           customer_city   str 0 (0.0%)          4119 Atributo Descriptivo
          customer_state   str 0 (0.0%)            27 Atributo Descriptivo

📋 GEOLOCATION
------------------------------------------------------------------------------------------------------------------------
                    Columna    Tipo    Nulos  Cardinalidad                  Rol
geolocation_zip_code_prefix   int64 0 (0.0%)         19015 Atributo Descriptivo
            geolocation_lat float64 0 (0.0%)        717360 Atributo Descriptivo
        